# IDM-VTON API for Kaggle GPU

Run this notebook on Kaggle with **Accelerator = GPU T4/P100** and **Internet = On**. It starts a FastAPI `/tryon` endpoint that the ASP.NET MVC website can call.


In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip -q install fastapi uvicorn python-multipart pillow numpy opencv-python-headless cloudpickle accelerate diffusers==0.25.0 transformers==4.36.2 huggingface_hub==0.20.3 safetensors einops gradio fvcore iopath omegaconf onnxruntime xformers==0.0.27.post2
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
!ls -lh /kaggle/working/cloudflared

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path('/kaggle/working/IDM-VTON')
if not repo.exists():
    subprocess.check_call(['git', 'clone', 'https://huggingface.co/spaces/yisol/IDM-VTON', str(repo)])

os.chdir(repo)
print('Working directory:', os.getcwd())

In [ ]:
%%writefile /kaggle/working/IDM-VTON/spaces.py
def GPU(fn=None, *args, **kwargs):
    if fn is None:
        return lambda f: f
    return fn


In [ ]:
from pathlib import Path

app_path = Path('/kaggle/working/IDM-VTON/app.py')
text = app_path.read_text()
text = text.replace('image_blocks.launch()', 'print("IDM-VTON components loaded for API mode")')
app_path.write_text(text)
print('Patched app.py to avoid launching Gradio UI during import')

## Load IDM-VTON

This downloads model files and loads the model into GPU memory. It can take several minutes.


In [ ]:
import importlib
import sys
import torch

sys.path.insert(0, '/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON/preprocess/openpose')
torch.cuda.empty_cache()
idm_app = importlib.import_module('app')
print('IDM-VTON loaded')

In [ ]:
%%writefile /kaggle/working/idm_vton_api.py
import base64
import io
import importlib
import sys

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

sys.path.insert(0, '/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON/preprocess/openpose')
idm_app = importlib.import_module('app')

api = FastAPI(title='IDM-VTON Kaggle API')
api.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


def normalize_clothing_type(value: str) -> str:
    value = (value or '').strip().lower().replace('_', '-').replace(' ', '-')
    return {
        'tee': 't-shirt',
        'tshirt': 't-shirt',
        't-shirts': 't-shirt',
        'sports-shirt': 'jersey',
        'sport-shirt': 'jersey',
        'hockey-jersey': 'jersey',
        'football-jersey': 'jersey',
        'basketball-jersey': 'jersey',
    }.get(value, value)


def garment_description(category: str, clothing_type: str, garment_view: str) -> str:
    kind = normalize_clothing_type(clothing_type or category)
    view = 'back view' if (garment_view or '').strip().lower() == 'back' else 'front view'
    if kind == 'jersey':
        return f'{view} oversized short sleeve sports jersey worn naturally on the upper body, preserve logo and number, collar on neck, sleeves on arms'
    if kind == 'shirt':
        return f'{view} button shirt worn naturally on the upper body, collar aligned to neck, sleeves on arms'
    if kind == 'hoodie':
        return f'{view} hoodie worn naturally on the upper body, hood behind neck, sleeves on arms'
    return f'{view} short sleeve round neck t-shirt worn naturally on the upper body, preserve garment design, collar below neck, sleeves on arms'


@api.get('/health')
def health():
    return {'status': 'ok', 'engine': 'idm-vton-kaggle'}


@api.post('/tryon')
async def tryon(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category: str = Form('upper'),
    clothing_type: str = Form(''),
    garment_view: str = Form('front'),
):
    person = Image.open(io.BytesIO(await person_image.read())).convert('RGB')
    garment = Image.open(io.BytesIO(await garment_image.read())).convert('RGB')

    result, _mask = idm_app.start_tryon(
        {'background': person, 'layers': [], 'composite': None},
        garment,
        garment_description(category, clothing_type, garment_view),
        True,
        True,
        30,
        42,
    )

    output = io.BytesIO()
    result.save(output, format='PNG')
    return {'outputImageBase64': base64.b64encode(output.getvalue()).decode('utf-8')}


In [ ]:
import re
import subprocess
import threading
import time

server = subprocess.Popen([
    'python', '-m', 'uvicorn', 'idm_vton_api:api', '--host', '0.0.0.0', '--port', '8000'
], cwd='/kaggle/working')
time.sleep(8)

public_url = None
for attempt in range(5):
    print(f'Tunnel attempt {attempt + 1}/5')
    tunnel = subprocess.Popen(
        ['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    for _ in range(120):
        line = tunnel.stdout.readline()
        print(line, end='')
        match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)
        if match:
            public_url = match.group(0)
            break

    if public_url:
        break
    tunnel.kill()
    time.sleep(8)

if not public_url:
    raise RuntimeError('Cloudflare did not create a public URL. Run this cell again.')

print('\nHealth URL:', public_url + '/health')
print('Paste this into VirtualTryOn__ApiUrl:')
print(public_url + '/tryon')

def keep_logs_open():
    for line in tunnel.stdout:
        print(line, end='')

threading.Thread(target=keep_logs_open, daemon=True).start()